### **Imports**

In [ ]:
import os
import pandas as pd
import spacy
import spacy.cli
import json
from pathlib import Path

from datasets import load_dataset
from utils import (
    normaliza_texto,
    filtrar_tokens_raros,
    construir_vocabulario,
    gerar_pares_skipgram
)

### **Datasets Direito**

#### **Normalização**

In [ ]:
# JurisTCU
dataset = load_dataset("LeandroRibeiro/JurisTCU", data_files="doc.csv")
df_juris_tcu = dataset["train"].to_pandas()
display(df_juris_tcu.head())

In [ ]:
display(df_juris_tcu["EXCERTO"].iloc[0])

In [ ]:
df_juris_tcu['texto'] = df_juris_tcu['EXCERTO'].apply(normaliza_texto)

In [ ]:
display(df_juris_tcu["texto"].iloc[0])

In [ ]:
# PortugueseLegalSentences
dataset = load_dataset("rufimelo/PortugueseLegalSentences-v3")
df_portuguese_legal_sentences = dataset["train"].to_pandas()
display(df_portuguese_legal_sentences.head())

In [ ]:
display(df_portuguese_legal_sentences['text'].iloc[0])

In [ ]:
df_portuguese_legal_sentences['texto'] = df_portuguese_legal_sentences['text'].apply(normaliza_texto)

In [ ]:
display(df_portuguese_legal_sentences['texto'].iloc[0])

In [ ]:
dataset = load_dataset("eduagarcia/oab_exams")
df_oab_exams = pd.DataFrame(dataset['train'])
display(df_oab_exams.head())

In [ ]:
display(df_oab_exams['question'].iloc[0])

In [ ]:
df_oab_exams['texto'] = df_oab_exams['question'].apply(normaliza_texto)

In [ ]:
display(df_oab_exams['texto'].iloc[0])

In [ ]:
# Concatenando apenas as colunas texto dos três dataframes
df_corpus_direito = pd.concat([
    df_juris_tcu['texto'],
    df_portuguese_legal_sentences['texto'],
    df_oab_exams['texto']
], ignore_index=True).to_frame(name="texto")

display(df_corpus_direito.head())

In [ ]:
df_corpus_direito.info()

#### **Tokenização**

In [ ]:
#Carregando o pacote e o texto
spacy.cli.download("pt_core_news_sm")
nlp = spacy.load("pt_core_news_sm")

In [ ]:
tokens_lista = []

print(f"Iniciando a tokenização de {len(df_corpus_direito)} linhas...")
print("Desativando componentes desnecessários para acelerar o processo...")

with nlp.select_pipes(disable=["parser", "ner", "lemmatizer"]):
    for doc in nlp.pipe(df_corpus_direito['texto'].astype(str), batch_size=128, n_process=-1):
        tokens_limpos = [token.text.lower() for token in doc if not token.is_punct and not token.is_space]
        tokens_lista.append(tokens_limpos)

In [ ]:
# Se for um DataFrame, cria a coluna:
df_corpus_direito["tokens"] = tokens_lista

print("Tokenização concluída com sucesso!")

#### **Filtragem Tokens Raros**

In [ ]:
# Filtragem de tokens raros
sentencas = df_corpus_direito["tokens"].tolist()

sentencas_filtradas, freqs = filtrar_tokens_raros(sentencas, min_freq=5)

df_corpus_direito["tokens_filtrados"] = sentencas_filtradas
df_corpus_direito["texto_filtrado"] = df_corpus_direito["tokens_filtrados"].apply(
    lambda x: " ".join(x))
df_corpus_direito["qtd_tokens"] = df_corpus_direito["tokens_filtrados"].apply(len)

display(df_corpus_direito[["texto", "texto_filtrado", "qtd_tokens"]].head(3))

#### **Construção do vocabulário**

In [ ]:
vocab, id2word = construir_vocabulario(sentencas_filtradas, vocab_size=2000, min_freq=5)

print(f"Tamanho do vocabulário: {len(vocab)}")
print(f"Exemplo de mapeamento token -> ID: {list(vocab.items())[:10]}")
print(f"Exemplo de mapeamento ID -> token: {list(id2word.items())[:10]}")

#### **Geração dos pares**

In [ ]:
pares = gerar_pares_skipgram(sentencas_filtradas, vocab, window_size=3)
pares_df = pd.DataFrame(pares, columns=["centro", "contexto"])
display(pares_df.head(3))

In [ ]:
display(pares_df)

#### **Salvamento dataset**

In [ ]:
base_dir = os.getcwd()
corpus_direito_path = os.path.join(base_dir,"..","datasets", "corpus_direito_tratado.csv")
print(f"Caminho para salvar o corpus tratado: {corpus_direito_path}")

pares_direito_path = os.path.join(base_dir,"..","datasets", "pares_skipgram_direito.csv")
print(f"Caminho para salvar os pares Skip-Gram: {pares_direito_path}")

# Salvando o df_corpus em um arquivo CSV
df_corpus_direito.to_csv(corpus_direito_path, index=False)
pares_df.to_csv(pares_direito_path, index=False)

#### **Salvando o vocabulário**

In [ ]:
output_dir = Path.cwd().parent / "datasets"
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / "vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False)

with open(output_dir / "id2word.json", "w", encoding="utf-8") as f:
    json.dump({str(k): v for k, v in id2word.items()}, f, ensure_ascii=False)

#### **Datasets Tecnologia**

#### **Normalização**

#### **Tokenização**

#### **Filtragem Tokens Raros**

#### **Construção do vocabulário**

#### **Salvamento dataset**

In [ ]:
corpus_ti_path = os.path.join(base_dir,"..","datasets", "corpus_tecnologia_tratado.csv")
print(f"Caminho para salvar o corpus tratado: {corpus_ti_path}")

# Salvando o df_corpus em um arquivo CSV
# df_corpus_ti.to_csv(corpus_ti_path, index=False)